# Do the seed budgets cut rankings on plateaus?

**Hypothesis.** Deterministic selectors break score ties by actor id, and MLTM itself is deterministic,
so the 30 repetitions of a deterministic selector are the same run. If a budget cut-off falls inside a
block of equally scored actors (a plateau), the seed set is an arbitrary, id-driven pick out of many
equivalent ones, and the reported result is a single sample whose spread we never see.
If cut-offs fall on steep parts of the sorted score curve, the seed set is well defined.

**Method.** For every network, selector (`data/rankings/v2`) and budget `b`, the cut-off is
`k = max(1, round(b * N))` (as in `sqs.pipeline.spreading`). At `k` we find the tie block, i.e. the
contiguous run of actors with the same score key as the `k`-th one, and report:

* `block`: size of the tie block, `inside`: how many of them are seeded, `left_out = block - inside`;
* `arbitrary`: `inside / k`, the share of the seed set picked by actor id rather than by score;
* `log10_sets`: `log10 C(block, inside)`, how many equally good seed sets exist.

The cut-off is safe iff `left_out == 0` (the block ends exactly at `k`).

Static SQ / SQ_R skip actors that are already active, so they read the ranking past `k`; `k` is a lower
bound. Because of this we also report the largest tie block within `[k, k + band]` (`band` = `BAND` · N).

`mds+deg` is ranked per repetition, so it is evaluated per rep, plus the mean pairwise Jaccard
similarity of its top-`k` sets across reps (how much variability the reps really have).

**Update after the fix (8d8cb48).** Deterministic selectors now have 30 rankings: rep 0 is the
original, reps 1..29 are replicas (`scripts/replicate_rankings.py`): the ranking shuffled with seed
`r`, then stably re-sorted by score, so ties no longer follow actor ids. Only the Static grid reads
them (`spreading.ranking_replicas`). On top of the tie analysis, the notebook now checks

1. that the replicas are valid (same scores, ties reordered only; for `deg-c-d` / `nghb-sd` also
   whether each replica is an order the discount process itself could have produced);
2. that the reps now seed different sets (top-`k` overlap across reps, every selector);
3. that this reaches the results (`data/static_results`): distinct seed sequences and spread of gain
   across reps, before vs after the fix, and whether the spread is large enough to change which
   selector wins.

Run the notebook top to bottom; it writes the PDF to `OUT_PDF`.

In [ ]:
import heapq
import json
from itertools import combinations
from math import comb, log10
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from network_diffusion.mln.centralities import degree, neighbourhood_size

from sqs.loaders.net_loader import load_network
from sqs.misc.ranking_replicas import SORT_KEYS
from sqs.runners.selectors.deterministic import _neighbours

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RANKINGS = ROOT / "data/rankings/v2"
OUT_PDF = ROOT / "data/static_results/ranking_plateaus.pdf"
RESULTS = ROOT / "data/static_results"
# optional summary of the results before the fix (same columns as ``summarise`` below)
BEFORE_CSV = ROOT / "data/static_results/results_before_fix.csv"

# configs/grids/static.yaml (configs/grids/strategies.yaml uses subsets of these)
BUDGETS = {"OR": [0.05, 0.1, 0.15, 0.2, 0.25], "AND": [0.15, 0.2, 0.25, 0.3, 0.35]}
ALL_BUDGETS = sorted({b for bs in BUDGETS.values() for b in bs})
SELECTORS = ["deg-c-d", "nghb-sd", "voterank", "ts-net", "mds+deg"]
BAND = 0.05  # fraction of N past the cut-off that SQ / SQ_R may still read
DISCOUNT = {"deg-c-d": degree, "nghb-sd": neighbourhood_size}  # replayed selectors

NETWORKS = sorted(p.name for p in RANKINGS.iterdir() if p.is_dir())
pd.set_option("display.width", 200, "display.max_rows", 400)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})
print(NETWORKS)

## Loading

Every selector now has a list of 30 rankings (rep 0 = original, then the replicas; `mds+deg`: one
search per rep). Each ranking becomes a frame with the plotted `value` and a hashable tie `key`, the
sort key the replicas use (`sqs.misc.ranking_replicas.SORT_KEYS`; voterank votes are compared to 4
decimals):

* `deg-c-d`, `nghb-sd`: discounted score when picked; `voterank`: votes when elected (0 for the unelected tail);
* `ts-net`: the lexicographic key of the four predicted outputs, plotted by `exposed`;
* `mds+deg`: `(driver, degree)`, plotted by degree (drivers come first).

In [ ]:
def _key_value(selector, score):
    if score is None:
        return None, np.nan
    key = SORT_KEYS[selector](score)
    if selector == "ts-net":
        return key, score["exposed"]
    if selector == "mds+deg":
        return key, score["degree"]
    return key, float(score)


def _path(network, selector, rep):
    if selector == "mds+deg":
        return RANKINGS / network / f"mds+deg__rep{rep}.json"
    if selector == "ts-net":
        files = [p for p in (RANKINGS / network).glob("ts-net__*.json") if "__rep" not in p.stem]
        assert len(files) == 1, files
        tag = files[0].stem
    else:
        tag = selector
    return RANKINGS / network / (f"{tag}.json" if rep == 0 else f"{tag}__rep{rep}.json")


def load(network, selector, rep):
    """Load one v2 ranking (original or replica) as a frame with actor, value and tie key."""
    entries = json.loads(_path(network, selector, rep).read_text())
    keys, values = zip(*(_key_value(selector, e["score"]) for e in entries))
    frame = pd.DataFrame({"actor": [e["actor"] for e in entries], "value": values})
    frame["key"] = list(keys)
    if selector == "mds+deg":
        frame["driver"] = [e["score"]["driver"] for e in entries]
    return frame


def reps_of(network, selector):
    """Return the repetitions that have a ranking of ``selector`` on ``network``."""
    reps = [0] if selector != "mds+deg" else []
    reps += sorted(
        int(p.stem.rsplit("rep", 1)[1])
        for p in (RANKINGS / network).glob("*__rep*.json")
        if p.stem.split("__")[0] == selector
    )
    return reps


rankings = {}  # (network, selector) -> list of frames, one per rep
for net in NETWORKS:
    for sel in SELECTORS:
        rankings[net, sel] = [load(net, sel, r) for r in reps_of(net, sel)]
sizes = {net: len(rankings[net, "deg-c-d"][0]) for net in NETWORKS}
pd.DataFrame(
    {sel: [len(rankings[net, sel]) for net in NETWORKS] for sel in SELECTORS}, index=NETWORKS
).assign(N=[sizes[n] for n in NETWORKS])

## Tie blocks at the cut-offs

In [ ]:
def blocks(keys):
    """Return (start, end) of the run of equal keys containing each position."""
    ids = np.zeros(len(keys), dtype=int)
    for i in range(1, len(keys)):
        ids[i] = ids[i - 1] + (keys[i] != keys[i - 1] or keys[i] is None)
    starts = pd.Series(np.arange(len(keys))).groupby(ids).transform("min").to_numpy()
    ends = pd.Series(np.arange(len(keys))).groupby(ids).transform("max").to_numpy() + 1
    return starts, ends


def cut_stats(frame, k, band):
    """Describe the tie block at cut-off ``k`` and the largest one ``band`` ranks past it."""
    starts, ends = blocks(list(frame["key"]))
    s, e = starts[k - 1], ends[k - 1]
    block, inside = e - s, k - s
    hi = min(len(frame), k + band)
    band_block = int((ends[k:hi] - starts[k:hi]).max()) if hi > k else 0
    return {
        "k": k,
        "block": block,
        "inside": inside,
        "left_out": block - inside,
        "arbitrary": inside / k if block > inside else 0.0,
        "log10_sets": log10(comb(block, inside)),
        "band_max_block": band_block,
        "safe": block == inside,
    }


rows = []
for (net, sel), ranking in rankings.items():
    n = sizes[net]
    for b in ALL_BUDGETS:
        k = max(1, round(b * n))
        protocols = "+".join(p for p, bs in BUDGETS.items() if b in bs)
        for rep, frame in enumerate(ranking):
            rows.append(
                {
                    "network": net,
                    "selector": sel,
                    "budget": b,
                    "protocols": protocols,
                    "rep": rep,
                    **cut_stats(frame, k, max(1, round(BAND * n))),
                }
            )
cuts = pd.DataFrame(rows)

# one row per (network, selector, budget), worst rep (replicas share the scores, so their tie
# blocks are the same; mds+deg differs per rep)
agg = cuts.groupby(["network", "selector", "budget", "protocols", "k"], as_index=False).agg(
    block=("block", "max"),
    left_out=("left_out", "max"),
    arbitrary=("arbitrary", "max"),
    arbitrary_mean=("arbitrary", "mean"),
    log10_sets=("log10_sets", "max"),
    band_max_block=("band_max_block", "max"),
    safe_share=("safe", "mean"),
)
agg["safe"] = agg["safe_share"] == 1.0
agg

## Are the replicas valid?

* `same_scores`: the replica has the same actors and the same score at every rank as the original
  (only tied actors are reordered);
* `deg-c-d`, `nghb-sd`: the discount process is replayed in the replica's order. A replica is a
  valid tie-break if every actor, when taken, has the highest current (discounted) score. Shuffling
  a finished ranking may break this, because which actor of a tie goes first changes the discounts
  of the next ones. Only the part the experiments read is replayed (`(max budget + BAND) * N`).
  `voterank` has the same issue (votes are recomputed after each election) but is not replayed.

In [ ]:
def first_invalid(net_obj, selector, actors, depth):
    """Return the first rank < ``depth`` taken without the top discounted score, else None."""
    scores = {a.actor_id: s for a, s in DISCOUNT[selector](net_obj).items()}
    by_id = {a.actor_id: a for a in net_obj.get_actors()}
    heap = [(-s, aid) for aid, s in scores.items()]
    heapq.heapify(heap)
    for pos, aid in enumerate(actors[:depth]):
        while heap[0][1] not in scores or -heap[0][0] != scores[heap[0][1]]:
            heapq.heappop(heap)  # stale entry
        if scores[aid] != -heap[0][0]:
            return pos
        del scores[aid]
        for nb in _neighbours(net_obj, by_id[aid]):
            if scores.get(nb):
                scores[nb] -= 1
                heapq.heappush(heap, (-scores[nb], nb))
    return None


rows = []
for net in NETWORKS:
    net_obj = load_network(net) if any(len(rankings[net, s]) > 1 for s in DISCOUNT) else None
    depth = min(sizes[net], round((max(ALL_BUDGETS) + BAND) * sizes[net]))
    for sel in SELECTORS:
        if sel == "mds+deg":
            continue
        original = rankings[net, sel][0]
        for rep, frame in enumerate(rankings[net, sel][1:], start=1):
            same = set(frame["actor"]) == set(original["actor"]) and list(frame["key"]) == list(
                original["key"]
            )
            actors = list(frame["actor"])
            bad = first_invalid(net_obj, sel, actors, depth) if sel in DISCOUNT else None
            rows.append(
                {
                    "network": net,
                    "selector": sel,
                    "rep": rep,
                    "same_scores": same,
                    "first_invalid": bad,
                    "first_invalid_frac": None if bad is None else bad / sizes[net],
                }
            )
validity = pd.DataFrame(rows)
validity_summary = validity.groupby(["selector", "network"]).agg(
    replicas=("rep", "size"),
    same_scores=("same_scores", "mean"),
    valid_share=("first_invalid", lambda x: x.isna().mean()),
    earliest_invalid=("first_invalid_frac", "min"),
)
validity_summary

## Do the reps now seed different sets?

Mean pairwise Jaccard similarity of the top-`k` sets of the 30 reps (1.0 = all reps seed the same set),
for every selector. For the replicated selectors it should drop below 1 exactly where the cut-off splits
a tie block.

In [ ]:
def mean_jaccard(frames, k):
    """Return the mean pairwise Jaccard similarity of the top-``k`` sets of ``frames``."""
    sets = [set(f["actor"].iloc[:k]) for f in frames]
    return float(np.mean([len(a & b) / len(a | b) for a, b in combinations(sets, 2)]))


overlap = pd.DataFrame(
    [
        {
            "network": net,
            "selector": sel,
            "budget": b,
            "jaccard": mean_jaccard(rankings[net, sel], max(1, round(b * sizes[net]))),
        }
        for net in NETWORKS
        for sel in SELECTORS
        for b in ALL_BUDGETS
    ]
)
check = agg.merge(overlap, on=["network", "selector", "budget"])
check = check[check.selector != "mds+deg"]
print("split tie block but identical sets:", int(((~check.safe) & (check.jaccard == 1)).sum()))
print("no split tie block but sets differ:", int((check.safe & (check.jaccard < 1)).sum()))
overlap.pivot_table(index="network", columns=["selector", "budget"], values="jaccard").round(2)

## Does it reach the results?

Per setting (network, protocol, mi, budget, selector, mode) of `data/static_results`: the number of
distinct seed sequences across reps (from `seedings`, so the seeding order counts too) and the spread
of `gain`. Before the fix every deterministic selector had one sequence and zero spread.

In [ ]:
KEYS = ["network", "protocol", "mi_value", "seed_budget", "ss_selector", "ss_mode"]


def summarise(root):
    """Per-setting reps, gain mean / std and distinct seed sequences of ``root``/*/results.csv."""
    parts = []
    for csv in sorted(root.glob("*/results.csv")):
        cols = [*KEYS, "rep", "gain", "seedings"]
        for chunk in pd.read_csv(csv, usecols=cols, chunksize=200_000):
            seeds = chunk.pop("seedings").fillna("")
            chunk["seed_hash"] = pd.util.hash_pandas_object(seeds, index=False)
            parts.append(chunk)
    df = pd.concat(parts)
    return df.groupby(KEYS, as_index=False).agg(
        reps=("rep", "nunique"),
        gain_mean=("gain", "mean"),
        gain_std=("gain", "std"),
        distinct_seedings=("seed_hash", "nunique"),
    )


after = summarise(RESULTS)
before = pd.read_csv(BEFORE_CSV) if BEFORE_CSV.exists() else None
after.groupby("ss_selector")[["reps", "distinct_seedings", "gain_std"]].agg(["mean", "max"])

In [ ]:
# winner margin vs run-to-run spread: does the spread blur which selector is best?
SETTING = ["network", "protocol", "mi_value", "seed_budget", "ss_mode"]


def margin(group):
    """Lead of the best selector over the runner-up and the larger of their gain stds."""
    top = group.sort_values("gain_mean", ascending=False).head(2)
    if len(top) < 2:
        return pd.Series({"lead": np.nan, "spread": np.nan})
    return pd.Series(
        {"lead": top.gain_mean.iloc[0] - top.gain_mean.iloc[1], "spread": top.gain_std.max()}
    )


margins = after.groupby(SETTING).apply(margin, include_groups=False).reset_index()
margins["blurred"] = margins["lead"] < margins["spread"]
blurred = margins.groupby("network")["blurred"].mean()
if before is not None:
    shift = after.merge(before, on=KEYS, suffixes=("", "_before"))
    shift["shift"] = (shift["gain_mean"] - shift["gain_mean_before"]).abs()
blurred.round(3)

## Figures

In [ ]:
BUDGET_COLORS = {"OR": "#1f77b4", "AND": "#d62728", "OR+AND": "#7b3fa0"}


def protocols_of(b):
    """Return the protocols that use budget ``b`` (e.g. 'OR+AND')."""
    return "+".join(p for p, bs in BUDGETS.items() if b in bs)


def plot_ranking(ax, net, sel):
    """Plot the sorted scores of one ranking with its budget cut-offs."""
    n = sizes[net]
    frames = rankings[net, sel] if sel == "mds+deg" else rankings[net, sel][:1]
    x = np.arange(1, n + 1) / n
    for i, frame in enumerate(frames):
        ax.plot(
            x,
            frame["value"],
            lw=0.6 if len(frames) > 1 else 1.0,
            color="0.2",
            alpha=0.25 if len(frames) > 1 else 1.0,
        )
    if sel == "mds+deg":
        drivers = [f["driver"].sum() / n for f in frames]
        ax.axvspan(
            min(drivers),
            max(drivers),
            color="0.85",
            lw=0,
            zorder=0,
            label="end of drivers (range over reps)",
        )
    frame = frames[0]
    starts, ends = blocks(list(frame["key"]))
    for b in ALL_BUDGETS:
        k = max(1, round(b * n))
        color = BUDGET_COLORS[protocols_of(b)]
        s, e = starts[k - 1], ends[k - 1]
        if e - s > k - s:  # straddled tie block
            ax.axvspan(s / n, e / n, color=color, alpha=0.18, lw=0)
        ax.axvline(k / n, color=color, lw=0.8, ls="-" if e - s == k - s else "--")
    ax.set_xlim(0, min(1.0, max(ALL_BUDGETS) + 2 * BAND))
    ax.set_title(sel)
    ax.set_xlabel("rank / N")
    ax.grid(alpha=0.3, lw=0.4)


def network_figure(net):
    """Return the figure with the rankings of all selectors on ``net``."""
    fig, axes = plt.subplots(1, len(SELECTORS), figsize=(16, 3.4))
    for ax, sel in zip(axes, SELECTORS):
        plot_ranking(ax, net, sel)
    axes[0].set_ylabel("score")
    handles = [
        plt.Line2D([], [], color=c, label=f"budget used by {p}") for p, c in BUDGET_COLORS.items()
    ]
    handles += [
        plt.Line2D([], [], color="0.3", ls="--", label="cut-off inside a tie block (shaded)")
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06)
    )
    fig.suptitle(f"{net} (N = {sizes[net]}): sorted scores with budget cut-offs", y=1.02)
    fig.tight_layout()
    return fig


def heatmap_figure(
    metric,
    title,
    fmt,
    vmax=None,
    data=None,
    cmap="Reds",
    vmin=0,
    selector="selector",
    budget="budget",
):
    """Return one heatmap of ``metric`` per selector (networks x budgets) of ``data`` (``agg``)."""
    data = agg if data is None else data
    fig, axes = plt.subplots(1, len(SELECTORS), figsize=(16, 4.2), sharey=True)
    for ax, sel in zip(axes, SELECTORS):
        rows = data[data[selector] == sel]
        table = rows.pivot_table(index="network", columns=budget, values=metric).reindex(NETWORKS)
        extra = (
            rows.pivot_table(index="network", columns=budget, values="block").reindex(
                index=NETWORKS, columns=table.columns
            )
            if "block" in rows
            else table
        )
        top = vmax or max(1e-9, data[metric].max())
        im = ax.imshow(table.to_numpy(), cmap=cmap, vmin=vmin, vmax=top, aspect="auto")
        for (i, j), v in np.ndenumerate(table.to_numpy()):
            if not np.isnan(v):
                ax.text(j, i, fmt(v, extra.to_numpy()[i, j]), ha="center", va="center", fontsize=6)
        ax.set_xticks(range(len(table.columns)), [f"{b:g}" for b in table.columns])
        ax.set_yticks(range(len(table.index)), table.index)
        ax.set_title(sel)
        ax.set_xlabel("budget")
    fig.colorbar(im, ax=axes, shrink=0.8, label=metric.replace("_", " "))
    fig.suptitle(title)
    return fig


FIGURES = []

CAPTIONS = {
    "arbitrary": """
        What it shows: each cell is one experimental setting (network x budget). The seeds
        are the top k actors of the ranking (k = budget x N). If the k-th actor has the same
        score as some actors just below it, the code cannot tell them apart and simply picks
        them in order of their ID. The top number is the share of seeds chosen this way; the
        bottom number (/m) is how many actors share that score.

        GOOD for us: an empty or pale cell. The seed set is decided by the scores alone.

        BAD for us: a dark cell. A large part of the seed set comes down to ID order, so the
        result is one arbitrary draw, and the 30 repetitions do not reveal it.

        Example: '0.26 /21' means 21 actors share a score, 13 of them get in and 8 do not;
        those 13 make up 26% of the 50 seeds. For mds+deg the cell shows the worst of the 30
        repetitions.
    """,
    "log10_sets": """
        What it shows: the same settings as the previous page, but counting how many
        different seed sets the scores would justify equally well. The cell shows the number
        of digits of that count (log10): 1 means about 10 sets, 5.3 about 200,000 sets.

        GOOD for us: an empty cell. There is only one correct seed set.

        BAD for us: a large number. Many seed sets are equally valid, yet the experiments
        only ever try one of them.

        This page tells the same story as the previous one; it simply makes the extreme cases
        (e.g. arxiv) easier to spot.
    """,
    "band_max_block": f"""
        What it shows: in the sequential modes (SQ, SQ_R), actors that are already active are
        skipped, so a run reaches further down the ranking than the budget line suggests. This
        page looks at the next {BAND:.0%} of ranks after the budget line and reports the
        largest group of actors sharing one score there.

        GOOD for us: an empty cell or a small number. Even if a run reads further, its choices
        are still driven by the scores.

        BAD for us: a large number. A run that reads that far will again choose among many
        identically scored actors by ID.

        Treat it as a warning sign rather than a precise measurement: we do not know exactly
        how far each run reads.
    """,
    "jaccard": """
        What it shows: for each setting we compare the seed sets (top k of the ranking) of the
        30 repetitions pair by pair and report their average overlap (1 = identical seeds,
        0.5 = about half shared). Since the fix, deg-c-d, nghb-sd, voterank and ts-net use a
        different random tie-break in every repetition; mds+deg has its own randomised search.

        GOOD for us: values below 1 wherever the first heatmap shows a split tie block, and 1
        where it does not. The repetitions now sample the seed sets that the scores cannot
        tell apart, and nothing else changes.

        BAD for us: values of exactly 1 next to a split tie block (the fix did not reach that
        ranking), or values below 1 where no tie is split (the replicas changed more than the
        ties).

        Note: ts-net hardly ever ties, so 1.00 is expected there. arxiv mds+deg scores 1.00
        everywhere: its search gives the same set in every repetition.
    """,
    "validity": """
        What it shows: the replicas were made by shuffling the finished ranking and sorting it
        by score again. For deg-c-d and nghb-sd we replay the discount process in each
        replica's order and check that every actor, when taken, really had the highest current
        score. Bars: share of replicas that pass up to the largest budget plus the band;
        the note gives the earliest rank at which any replica fails.

        GOOD for us: 100%. Each replica is a ranking the selector itself could have produced
        with a different tie-break.

        BAD for us: low shares, especially with a failure before the budgets (rank < 0.35*N).
        Then some replicas pick an actor whose score had already been discounted, i.e. an
        order the method would never produce. The variability is then partly artificial.

        voterank has the same risk (votes are recomputed after every election) but is not
        replayed here.
    """,
    "distinct": """
        What it shows: from the actual simulation results (data/static_results), the number of
        different seed sequences (which actors, in which order) used by the 30 repetitions of
        each setting, averaged over protocol, mi and mode. Before the fix it was 1 for every
        selector except mds+deg.

        GOOD for us: values above 1 for deg-c-d, nghb-sd and voterank where the rankings have
        split ties. The fix reaches the simulations and the repetitions now differ.

        BAD for us: 1 across the board for a selector with ties: the experiments still use a
        single seed sequence.

        Values can exceed what the ties at the budget line suggest, because SQ and SQ_R read
        deeper and the order of seeding counts too.
    """,
    "gain_std": """
        What it shows: the standard deviation of gain across the 30 repetitions, averaged over
        protocol, mi and mode. This is the run-to-run spread that the id-based tie-break
        used to hide (it was exactly 0 for all selectors except mds+deg).

        GOOD for us: small values. Tie-breaking barely changes the outcome, so earlier
        conclusions are likely to stand.

        BAD for us: large values compared with the differences between selectors. Then the
        single run reported before could have been a lucky or unlucky draw.
    """,
    "shift": """
        What it shows: how much the mean gain of each setting moved after the fix
        (|mean after - value before|), averaged over protocol, mi and mode. Before the fix
        every repetition gave the same value, so this is the bias of the id-based tie-break.

        GOOD for us: empty or pale cells. The old single run was representative.

        BAD for us: dark cells. The old results were systematically off for that selector and
        network, so comparisons drawn from them need checking.
    """,
    "blurred": """
        What it shows: for each setting (network, protocol, mi, budget, mode) we take the
        selector with the highest mean gain and the runner-up, and check whether the lead is
        smaller than the run-to-run spread (the larger of their two standard deviations).
        Bars: share of settings where that happens.

        GOOD for us: low shares. The best selector stays clearly ahead even though the
        repetitions now vary.

        BAD for us: high shares. In many settings the winner is not clearly separated from the
        runner-up, so claims about which method is best need a statistical test, not just
        mean values.
    """,
    "network": """
        What it shows: actors sorted from best (left) to worst (right) by the score each
        method gives them. The vertical lines are the budgets used in the experiments (blue:
        OR only, red: AND only, purple: both).

        GOOD for us: a solid line where the curve is steep. Actors just inside and just
        outside the budget have clearly different scores, so the seed set is well defined.

        BAD for us: a dashed line inside a shaded band, where the curve is flat. Every actor
        in the band has the same score and the budget splits them, so who becomes a seed is
        decided by ID.

        mds+deg: one faint curve per repetition (darker where they agree). Drivers come first
        and the remaining actors second, each group sorted by degree, which is why the curve
        jumps back up where the drivers end (grey band). Its budget lines and shading are for
        repetition 0 only.

        Scores: deg-c-d and nghb-sd: discounted score when chosen; voterank: votes when
        elected; ts-net: predicted 'exposed'.
    """,
}


def caption(fig, text, y=-0.02):
    """Put ``text`` under ``fig``, one wrapped paragraph per blank-line-separated block."""
    import textwrap  # noqa: PLC0415

    width = int(fig.get_figwidth() * 15)
    pars = textwrap.dedent(text).strip().split("\n\n")
    lines = [textwrap.fill(" ".join(par.split()), width) for par in pars]
    fig.text(0.01, y, "\n".join(lines), ha="left", va="top", fontsize=8, linespacing=1.4)
    return fig

In [ ]:
fig = heatmap_figure(
    "arbitrary",
    "Share of the seed set picked by actor id at the cut-off "
    "(cell: share / tie block size; mds+deg: worst rep)",
    lambda v, blk: "" if v == 0 else f"{v:.2f}\n/{int(blk)}",
    vmax=1.0,
)
FIGURES.append(caption(fig, CAPTIONS["arbitrary"]))

In [ ]:
fig = heatmap_figure(
    "log10_sets",
    "log10 of the number of equally good seed sets at the cut-off (0 = unique)",
    lambda v, blk: "" if v == 0 else f"{v:.1f}",
)
FIGURES.append(caption(fig, CAPTIONS["log10_sets"]))

In [ ]:
fig = heatmap_figure(
    "band_max_block",
    f"Largest tie block within {BAND:g}·N past the cut-off (what SQ / SQ_R may read further on)",
    lambda v, blk: "" if v <= 1 else f"{int(v)}",
)
FIGURES.append(caption(fig, CAPTIONS["band_max_block"]))

In [ ]:
fig = heatmap_figure(
    "jaccard",
    "Overlap of the top-k seed sets across the 30 reps (mean pairwise Jaccard; 1 = identical)",
    lambda v, _: f"{v:.2f}",
    vmax=1.0,
    data=overlap,
    cmap="viridis",
)
FIGURES.append(caption(fig, CAPTIONS["jaccard"]))

In [ ]:
fig, axes = plt.subplots(1, len(DISCOUNT), figsize=(9, 4.2), sharey=True)
for ax, sel in zip(axes, DISCOUNT):
    table = validity_summary.loc[sel].reindex(NETWORKS)
    ax.barh(range(len(NETWORKS)), table["valid_share"], color="#4c72b0")
    for i, (share, earliest) in enumerate(zip(table["valid_share"], table["earliest_invalid"])):
        note = "" if np.isnan(earliest) else f"  first invalid at {earliest:.2f}·N"
        ax.text(min(share, 1.0), i, f" {share:.0%}{note}", va="center", fontsize=6)
    ax.set_yticks(range(len(NETWORKS)), NETWORKS)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.6)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_xlabel("share of replicas that are a valid order")
    ax.set_title(sel)
fig.suptitle(
    f"Replica validity up to {max(ALL_BUDGETS) + BAND:.2f}·N "
    f"(all replicas keep the original scores: {bool(validity.same_scores.all())})"
)
fig.tight_layout()
FIGURES.append(caption(fig, CAPTIONS["validity"], y=-0.03))

In [ ]:
fig = heatmap_figure(
    "distinct_seedings",
    "Distinct seed sequences across the 30 reps "
    "(mean over protocol, mi and mode; 1 = all reps identical)",
    lambda v, _: f"{v:.0f}" if v >= 1.5 else ("1" if v == 1 else f"{v:.1f}"),
    vmax=30,
    vmin=1,
    data=after,
    cmap="Greens",
    selector="ss_selector",
    budget="seed_budget",
)
FIGURES.append(caption(fig, CAPTIONS["distinct"]))

In [ ]:
fig = heatmap_figure(
    "gain_std",
    "Spread of gain across the 30 reps (std, mean over protocol, mi and mode)",
    lambda v, _: "" if v == 0 else f"{v:.1f}",
    data=after,
    cmap="Oranges",
    selector="ss_selector",
    budget="seed_budget",
)
FIGURES.append(caption(fig, CAPTIONS["gain_std"]))

In [ ]:
if before is not None:
    fig = heatmap_figure(
        "shift",
        "Change of mean gain after the fix (|after - before|, mean over protocol, mi and mode)",
        lambda v, _: "" if v < 0.05 else f"{v:.1f}",
        data=shift,
        cmap="Purples",
        selector="ss_selector",
        budget="seed_budget",
    )
    FIGURES.append(caption(fig, CAPTIONS["shift"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(range(len(blurred)), blurred.to_numpy(), color="#c44e52")
for i, v in enumerate(blurred.to_numpy()):
    ax.text(v, i, f" {v:.0%}", va="center", fontsize=7)
ax.set_yticks(range(len(blurred)), blurred.index)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("share of settings where the winner's lead < run-to-run spread")
ax.set_title("Is the best selector still clear once reps vary?")
fig.tight_layout()
FIGURES.append(caption(fig, CAPTIONS["blurred"], y=-0.04))

In [ ]:
for net in NETWORKS:
    FIGURES.append(caption(network_figure(net), CAPTIONS["network"], y=-0.13))

## Verdict

In [ ]:
unsafe = agg[~agg.safe]
summary = (
    agg.assign(unsafe=~agg.safe)
    .groupby("selector")
    .agg(
        cells=("unsafe", "size"),
        unsafe=("unsafe", "sum"),
        worst_arbitrary=("arbitrary", "max"),
        mean_arbitrary=("arbitrary", "mean"),
    )
    .reindex(SELECTORS)
)
lines = [
    f"Cut-offs checked: {len(agg)} ({len(NETWORKS)} networks x {len(SELECTORS)} selectors "
    f"x {len(ALL_BUDGETS)} budgets).",
    f"Cut-offs inside a tie block: {len(unsafe)} ({len(unsafe) / len(agg):.0%}).",
    "",
    "Per selector (arbitrary = share of seeds picked by actor id; mds+deg = worst rep):",
]
for sel, r in summary.iterrows():
    lines.append(
        f"  {sel:9s} unsafe {int(r.unsafe):3d}/{int(r.cells)}  "
        f"worst {r.worst_arbitrary:.2f}  mean {r.mean_arbitrary:.3f}"
    )
lines += ["", "Worst cut-offs:"]
for _, r in agg.sort_values("arbitrary", ascending=False).head(15).iterrows():
    if r.arbitrary == 0:
        break
    lines.append(
        f"  {r.network:30s} {r.selector:9s} b={r.budget:<5g} k={r.k:<6d} block={r.block:<6d} "
        f"left_out={r.left_out:<6d} arbitrary={r.arbitrary:.2f}  log10(sets)={r.log10_sets:.1f}"
    )
mds = overlap[overlap.selector == "mds+deg"].jaccard.to_numpy()
lines += [
    "",
    "mds+deg mean pairwise Jaccard of top-k across reps: "
    f"min {mds.min():.2f}, median {np.median(mds):.2f}, max {mds.max():.2f}.",
]
det = after[after.ss_selector != "mds+deg"]
lines += [
    "",
    "After the fix (replicas, data/static_results):",
    f"  replicas keep the original scores: {bool(validity.same_scores.all())}",
]
for sel in DISCOUNT:
    v = validity[validity.selector == sel]
    lines.append(
        f"  {sel}: {v.first_invalid.isna().mean():.0%} of replicas are a valid discount order "
        f"up to {max(ALL_BUDGETS) + BAND:.2f}*N"
    )
lines += [
    f"  split tie but identical top-k sets: {int(((~check.safe) & (check.jaccard == 1)).sum())}, "
    f"no split tie but different sets: {int((check.safe & (check.jaccard < 1)).sum())}",
    "  settings with more than one seed sequence across reps (deterministic selectors): "
    f"{(det.distinct_seedings > 1).mean():.0%}",
    f"  mean gain std (deterministic selectors): {det.gain_std.mean():.2f}, "
    f"max {det.gain_std.max():.2f}",
    f"  settings where the winner's lead < spread: {margins.blurred.mean():.0%}",
]
if before is not None:
    old_distinct = before[before.ss_selector != "mds+deg"].distinct_seedings.mean()
    det_shift = shift[shift.ss_selector != "mds+deg"]["shift"]
    lines.append(
        f"  before the fix: mean distinct sequences {old_distinct:.1f}; "
        f"mean |gain shift| {det_shift.mean():.2f}, max {det_shift.max():.2f}"
    )
VERDICT = "\n".join(lines)
print(VERDICT)

In [ ]:
def text_page(title, body):
    """Return a figure with a title and monospace text, used as a PDF page."""
    fig = plt.figure(figsize=(11.7, 8.3))
    fig.text(0.05, 0.95, title, fontsize=14, weight="bold", va="top")
    fig.text(0.05, 0.90, body, fontsize=8, family="monospace", va="top")
    return fig


INTRO = (
    "Fix under test (8d8cb48): deterministic selectors get 29 replicas with random tie-breaks\n"
    "(rep r of the Static grid uses replica r).\n\n"
    "Hypothesis: deterministic selectors break ties by actor id and MLTM is deterministic,\n"
    "so all reps of a deterministic selector are the same run. A budget cut-off inside a\n"
    "block of equally scored actors makes the seed set an id-driven pick among many\n"
    "equivalent ones (variance we never observe).\n\n"
    "Cut-off k = max(1, round(b * N)). Tie block = contiguous run of actors whose score key\n"
    "equals the k-th one (keys of sqs.misc.ranking_replicas.SORT_KEYS; voterank to 4\n"
    "decimals; ts-net: full lexicographic key; mds+deg: (driver, degree)).\n"
    "  arbitrary  = seeds inside the block / k   (0 -> the block ends at k -> safe)\n"
    "  log10_sets = log10 C(block, inside)       (number of equally good seed sets)\n"
    f"  band       = largest block within {BAND:g}*N past k (Static SQ / SQ_R skip active\n"
    "               actors and read deeper than k, so k is only a lower bound of the real\n"
    "               cut-off)\n\n"
    f"Budgets: OR {BUDGETS['OR']}, AND {BUDGETS['AND']}.\n"
    f"Rankings: {RANKINGS.relative_to(ROOT)}\n\n"
)
OUT_PDF.parent.mkdir(parents=True, exist_ok=True)
with PdfPages(OUT_PDF) as pdf:
    pdf.savefig(
        text_page("Do the seed budgets cut rankings on plateaus?", INTRO + VERDICT),
        bbox_inches="tight",
    )
    for fig in FIGURES:
        pdf.savefig(fig, bbox_inches="tight")
    table = unsafe.sort_values(["selector", "network", "budget"])[
        [
            "network",
            "selector",
            "budget",
            "protocols",
            "k",
            "block",
            "left_out",
            "arbitrary",
            "log10_sets",
            "band_max_block",
        ]
    ]
    per_page = 60
    for i in range(0, max(1, len(table)), per_page):
        chunk = table.iloc[i : i + per_page].to_string(
            index=False, float_format=lambda v: f"{v:.3g}"
        )
        pdf.savefig(
            text_page(
                f"Unsafe cut-offs ({i + 1}-{min(i + per_page, len(table))} of {len(table)})", chunk
            ),
            bbox_inches="tight",
        )
print(f"written {OUT_PDF}")